# Phase 9 — CNN Model: Dataset & First Training Run
## AI-Based Crop Disease Detection and Smart Farming Guidance System — Arecanut belt (Dakshina Kannada / Udupi / Shivamogga) mixed-cropping system

**Run this in Google Colab** with a GPU runtime: `Runtime > Change runtime type > T4 GPU`.

**What this notebook does, in order:**
1. Downloads real disease-image datasets for **Rice, Black Pepper, and Cocoa** from Kaggle — the three
   crops in this region's system that have public datasets. (Arecanut has no public dataset yet — see
   the note at the end of this notebook; that's a real research opportunity, not a gap we're hiding.)
2. Has you inspect the real downloaded folder structure and fill in a small class-mapping config —
   see Section 2 for why this step can't be automated blindly.
3. Splits the mapped data into train/validation/test sets.
4. Builds a data pipeline with augmentation (rotation, flip, zoom).
5. Builds a transfer-learning model: MobileNetV2 (ImageNet-pretrained) with a new classification head, base frozen.
6. Trains the head, then optionally fine-tunes the top of the base network.
7. Evaluates on a held-out test set — **real** accuracy/precision/recall/F1/confusion matrix, not estimated.
8. Saves the trained model and its class-label mapping, versioned (`model_v1.h5`, `labels_v1.json`).

Every number this notebook reports comes from an actual run — if you re-run it, your numbers may differ slightly (random split, random initialization of the head, GPU non-determinism), and that's expected. Report whatever you actually get, not these placeholder expectations.


## 1. Setup

Colab already has TensorFlow. We only need a few extra packages for the metrics/plots and for a clean train/val/test split.

In [ ]:
!pip install -q split-folders scikit-learn seaborn

import os, json, shutil, random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    print("\nWARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU, "
          "then Runtime > Restart session, and re-run from the top. Training will be "
          "very slow on CPU.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 2. Download the datasets — Rice, Black Pepper, Cocoa

Unlike the old plan (one GitHub repo covering every crop), these three real, regionally-relevant
datasets each live in a separate Kaggle dataset, and each was uploaded by a different person with
their own folder layout. There's no way to know the exact internal folder names in advance without
actually downloading and looking — so this section has **one manual step**: after downloading, you'll
see the real folder tree printed, and you fill in a short mapping from "this folder" to "this class
label". This is a one-time ~10 minute step, not a fragile guess baked into the code.

**You'll need a free Kaggle account and API token:**
1. Go to [kaggle.com/settings](https://www.kaggle.com/settings) → **API Tokens** → **Generate New Token**.
2. Copy the token string shown (starts with `KGAT_`) — Kaggle only shows it once, so grab it now.
3. Run the cell below, and paste the token into the box that appears (input is hidden as you type).

In [ ]:
import getpass, os

kaggle_token = getpass.getpass("Paste your Kaggle API token (KGAT_...): ")
os.environ["KAGGLE_API_TOKEN"] = kaggle_token.strip()

!pip install -q "kaggle>=1.8.0"
!kaggle datasets list -s rice > /dev/null && print("Kaggle API ready — token works.")


Now download the three datasets. One Kaggle handle per crop — chosen from the dataset survey in
`PROJECT_SCOPE_PIVOT.md` for being the most complete/usable option for that crop.

In [ ]:
DATASETS = {
    "Rice":  "nirmalsankalana/rice-leaf-disease-image",
    "Pepper": "adithyantg/black-pepper-leaf-disease-mini-dataset",
    "Cocoa": "bryandarquea/cocoa-diseases",
}

RAW_DOWNLOAD_ROOT = "kaggle_raw"
shutil.rmtree(RAW_DOWNLOAD_ROOT, ignore_errors=True)
os.makedirs(RAW_DOWNLOAD_ROOT, exist_ok=True)

for crop, handle in DATASETS.items():
    dest = os.path.join(RAW_DOWNLOAD_ROOT, crop)
    os.makedirs(dest, exist_ok=True)
    print(f"\n=== Downloading {crop}: {handle} ===")
    !kaggle datasets download -d {handle} -p {dest} --unzip


### Inspect the real folder structure

This prints every folder that directly contains image files, for each crop, with how many images are
in it. **Read this output carefully before the next cell** — the exact names depend on how each
Kaggle uploader organized their files (could be `train/Blast/`, `Healthy/`, `bacterial_leaf_blight/`,
anything).

In [ ]:
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")

def find_image_dirs(root, base):
    """Return every directory under `root` that directly contains image files, with counts.
    Paths are reported relative to `base` (so they include the crop-name prefix and can be
    pasted straight into CLASS_FOLDER_MAP below)."""
    found = []
    for dirpath, _dirnames, filenames in os.walk(root):
        images = [f for f in filenames if f.endswith(IMAGE_EXTS)]
        if images:
            rel = os.path.relpath(dirpath, base)
            found.append((rel, len(images)))
    return sorted(found)

for crop in DATASETS:
    print(f"\n=== {crop} — folders containing images ===")
    for rel, n in find_image_dirs(os.path.join(RAW_DOWNLOAD_ROOT, crop), RAW_DOWNLOAD_ROOT):
        print(f"  {rel!r}: {n} images")


### Fill in the class mapping

Copy the exact folder paths printed above (the `'...'` strings) into `CLASS_FOLDER_MAP` below, one
entry per line, mapped to a clean class label in the `Crop___Condition` format (matching the style the
rest of this project uses). **Skip folders that are just a redundant train/test split of the same
classes** — we do our own train/val/test split in Section 3, so only map each real class once (pick
whichever split folder has the most images for that class, or merge — your call, just don't double count).

If a dataset has a `Healthy` folder, map it too — healthy images are as important as disease images for
a real classifier.

In [ ]:
CLASS_FOLDER_MAP = {
    # Example — replace these with the REAL paths from the printed output above:
    # "Rice/train/Bacterialblight": "Rice___Bacterial_Leaf_Blight",
    # "Rice/train/Blast":           "Rice___Blast",
    # "Rice/train/Healthy":         "Rice___Healthy",
    # "Pepper/Foot_Rot":            "Pepper___Foot_Rot",
    # "Pepper/Healthy":             "Pepper___Healthy",
    # "Cocoa/Black_Pod":            "Cocoa___Black_Pod",
    # "Cocoa/Healthy":              "Cocoa___Healthy",
}

assert CLASS_FOLDER_MAP, (
    "CLASS_FOLDER_MAP is empty — fill it in using the folder paths printed in the cell above "
    "before continuing."
)
print(f"{len(CLASS_FOLDER_MAP)} classes mapped:")
for k, v in CLASS_FOLDER_MAP.items():
    print(f"  {k!r} -> {v}")


In [ ]:
# Copy a capped number of images per mapped class into a clean, flat dataset_raw/ folder
MAX_IMAGES_PER_CLASS = 400  # caps class size for a faster, still-real first training run

DATASET_ROOT = "dataset_raw"
shutil.rmtree(DATASET_ROOT, ignore_errors=True)
os.makedirs(DATASET_ROOT, exist_ok=True)

class_counts = {}
for rel_path, label in CLASS_FOLDER_MAP.items():
    # rel_path is relative to RAW_DOWNLOAD_ROOT, exactly as printed above (crop-name prefix included).
    src_dir = os.path.join(RAW_DOWNLOAD_ROOT, rel_path)
    if not os.path.isdir(src_dir):
        raise FileNotFoundError(
            f"{src_dir!r} does not exist — double-check the path you copied from the printed tree above."
        )
    files = sorted(f for f in os.listdir(src_dir) if f.endswith(IMAGE_EXTS))
    random.shuffle(files)
    files = files[:MAX_IMAGES_PER_CLASS]
    dst_dir = os.path.join(DATASET_ROOT, label)
    os.makedirs(dst_dir, exist_ok=True)
    for f in files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(dst_dir, f))
    class_counts[label] = len(files)

print("Images per class (after capping):")
for cls, n in class_counts.items():
    print(f"  {cls}: {n}")
print(f"\nTotal images: {sum(class_counts.values())}")


## 3. Train / validation / test split

70% train, 15% validation, 15% test, stratified per class by `split-folders`.

In [ ]:
import splitfolders

SPLIT_ROOT = "dataset_split"
shutil.rmtree(SPLIT_ROOT, ignore_errors=True)
splitfolders.ratio(DATASET_ROOT, output=SPLIT_ROOT, seed=SEED, ratio=(0.7, 0.15, 0.15))

for split in ["train", "val", "test"]:
    n = sum(len(files) for _, _, files in os.walk(os.path.join(SPLIT_ROOT, split)))
    print(f"{split}: {n} images")


## 4. Data pipeline with augmentation

Augmentation (rotation, flip, zoom) is applied only to the training set — validation and test data
stay unmodified so evaluation reflects real, unaltered images.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{SPLIT_ROOT}/train", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical", seed=SEED
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{SPLIT_ROOT}/val", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical", seed=SEED
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{SPLIT_ROOT}/test", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical", seed=SEED, shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"{num_classes} classes:", class_names)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
], name="augmentation")

AUTOTUNE = tf.data.AUTOTUNE
preprocess = tf.keras.applications.mobilenet_v2.preprocess_input

def prep_train(x, y):
    x = data_augmentation(x)
    x = preprocess(x)
    return x, y

def prep_eval(x, y):
    x = preprocess(x)
    return x, y

train_ds_p = train_ds.map(prep_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds_p = val_ds.map(prep_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds_p = test_ds.map(prep_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


## 5. Build the model — MobileNetV2 transfer learning

The pretrained base is frozen first (train only the new head), then optionally fine-tuned in the next
section. Freezing first prevents the randomly-initialized head from wrecking the pretrained weights
with large early gradients.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


## 6. Train the head (base frozen)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
]

EPOCHS = 15
history = model.fit(
    train_ds_p,
    validation_data=val_ds_p,
    epochs=EPOCHS,
    callbacks=callbacks,
)


## 7. Fine-tune (optional but recommended)

Unfreeze the top layers of the MobileNetV2 base and continue training at a much lower learning rate.
This usually improves accuracy meaningfully over the frozen-base run — worth doing for the number you
actually report to your guide, but the frozen-base run above is already a legitimate first result if
you're short on time.

In [ ]:
base_model.trainable = True

# Freeze everything except the last ~30 layers of the base network.
FINE_TUNE_AT = len(base_model.layers) - 30
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

FT_EPOCHS = 10
history_fine = model.fit(
    train_ds_p,
    validation_data=val_ds_p,
    epochs=FT_EPOCHS,
    callbacks=callbacks,
)


## 8. Evaluate on the held-out test set

This is the number that matters — the model has never seen these images, augmented or otherwise.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds_p)
print(f"\nTest accuracy: {test_accuracy:.4f}")
print(f"Test loss: {test_loss:.4f}")

y_true = []
y_pred = []
for images, labels in test_ds_p:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n=== Classification Report (real numbers from this run) ===")
report = classification_report(y_true, y_pred, target_names=class_names, digits=3)
print(report)

with open("training_report.md", "w") as f:
    f.write("# Phase 9 — Training Report\n\n")
    f.write(f"Test accuracy: {test_accuracy:.4f}\n\n")
    f.write(f"Test loss: {test_loss:.4f}\n\n")
    f.write("## Classification Report\n\n```\n" + report + "\n```\n")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(max(8, num_classes * 0.6), max(6, num_classes * 0.5)))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Phase 9 First Training Run")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()


## 9. Save the model and label mapping

Versioned so later phases (and later retraining runs) don't silently overwrite this baseline.

In [ ]:
MODEL_VERSION = "v1"

model.save(f"model_{MODEL_VERSION}.h5")

label_map = {i: name for i, name in enumerate(class_names)}
with open(f"labels_{MODEL_VERSION}.json", "w") as f:
    json.dump(label_map, f, indent=2)

print(f"Saved model_{MODEL_VERSION}.h5 and labels_{MODEL_VERSION}.json")
print("Label mapping:", label_map)


## 10. Download your results

Run the cell below to download the four files this notebook produces. Put all four into your
project's `ml-training/` folder (next to this notebook) — Phase 10 (AI Integration) will load
`model_v1.h5` and `labels_v1.json` from there to wire real predictions into the Django backend.

In [ ]:
from google.colab import files

for fname in ["model_v1.h5", "labels_v1.json", "training_report.md", "confusion_matrix.png"]:
    if os.path.exists(fname):
        files.download(fname)
    else:
        print(f"Missing: {fname} — check the cells above ran without error.")


---
## Notes for your guide meeting

- This is a **first training run**, not a final model — Phase 9's job is to prove the pipeline works
  end-to-end and produce real, honestly-reported numbers, not to hit a target accuracy.
- This model covers **Rice, Black Pepper, and Cocoa** — three of the four crops in the region's real
  mixed-cropping system (arecanut palm as the backbone, pepper trellised on it, cocoa shaded beneath,
  rice in the valley paddies). Framing this honestly to your guide: it's a multi-crop model evaluated
  as one coherent system, not four disconnected single-crop demos — that's a deliberate design choice
  worth stating explicitly, not an accident of what data was available.
- **Arecanut is deliberately not in this first model** — no public image dataset exists for it (verified
  during the dataset survey in `PROJECT_SCOPE_PIVOT.md`). If you or a teammate can collect even
  150–300 real arecanut leaf photos (healthy + 2–3 common conditions like Yellow Leaf Disease) from a
  local garden or KVK/agricultural extension office, that's a genuine, citable research contribution —
  "to our knowledge, the first published image dataset for arecanut disease classification" — publishable
  alongside the paper on Kaggle or Mendeley Data (the same way the RoCoLe coffee-leaf dataset was). This
  is the strongest path to something reviewers will see as actually new, not just an app.
- Report exactly what `training_report.md` and `confusion_matrix.png` say — if a class has low
  precision/recall, that's worth naming honestly (e.g. a class with too few or visually similar
  images), not smoothing over.
- Phase 10 connects this saved model to the `ScanUploadView` endpoint built in Phase 8, replacing the
  current "Processing not yet connected" placeholder with a real prediction.
